In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
base = '/content/drive/MyDrive/proyecto_reciclaje'
for carpeta in ['organico', 'reciclable', 'no_reciclable']:
    ruta = os.path.join(base, carpeta)
    print(carpeta, '→', len(os.listdir(ruta)), 'fotos')

organico → 15 fotos
reciclable → 15 fotos
no_reciclable → 15 fotos


In [ ]:
import tensorflow as tf

base = '/content/drive/MyDrive/proyecto_reciclaje'
IMG_SIZE = 160

dataset = tf.keras.utils.image_dataset_from_directory(
    base,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16,
    seed=42
)
print('Clases:', dataset.class_names)

Found 45 files belonging to 3 classes.
Clases: ['no_reciclable', 'organico', 'reciclable']


In [ ]:
from tensorflow.keras.applications import mobilenet_v2

def preparar(imagen, etiqueta):
    return mobilenet_v2.preprocess_input(imagen), etiqueta

ds = dataset.map(preparar)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

modelo = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(3, activation='softmax')  # CAMBIO 1: 3 clases, no 1
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',  # CAMBIO 2: ya no es binario
    metrics=['accuracy']
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
historia = modelo.fit(ds, epochs=5)

Epoch 1/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.2889 - loss: 1.5508
Epoch 2/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 231ms/step - accuracy: 0.3333 - loss: 1.6047
Epoch 3/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 245ms/step - accuracy: 0.3556 - loss: 1.3629
Epoch 4/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 264ms/step - accuracy: 0.2667 - loss: 1.5400
Epoch 5/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 257ms/step - accuracy: 0.2889 - loss: 1.3473


In [ ]:
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

historia = modelo.fit(ds, epochs=15)

Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 418ms/step - accuracy: 0.2889 - loss: 1.4005
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 248ms/step - accuracy: 0.5556 - loss: 0.9719
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 288ms/step - accuracy: 0.5778 - loss: 0.8230
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 255ms/step - accuracy: 0.7778 - loss: 0.5302
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 249ms/step - accuracy: 0.8444 - loss: 0.4534
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 245ms/step - accuracy: 0.9333 - loss: 0.3227
Epoch 7/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 266ms/step - accuracy: 0.9111 - loss: 0.3079
Epoch 8/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 452ms/step - accuracy: 0.9778 - loss: 0.1817
Epoch 9/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 448ms/step - accuracy: 1.0000 - loss: 0.1268
Epoch 10/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 246ms/step - accuracy: 1.0000 - loss: 0.1183
Epoch 11/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 229ms/step - accuracy: 1.0000 - loss: 0.1030
Epoch 12/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 255ms/step - accuracy: 1.0000 - lo

In [ ]:
dataset_train = tf.keras.utils.image_dataset_from_directory(
    base, validation_split=0.2, subset='training', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=16)

dataset_val = tf.keras.utils.image_dataset_from_directory(
    base, validation_split=0.2, subset='validation', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=16)

Found 45 files belonging to 3 classes.
Using 36 files for training.
Found 45 files belonging to 3 classes.
Using 9 files for validation.


In [ ]:
ds_train = dataset_train.map(preparar)
ds_val = dataset_val.map(preparar)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

modelo = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(3, activation='softmax')
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

historia = modelo.fit(ds_train, epochs=15)

print('--- EXAMEN FINAL (validación) ---')
resultado = modelo.evaluate(ds_val)
print('Accuracy real:', round(resultado[1]*100, 1), '%')

Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 293ms/step - accuracy: 0.2778 - loss: 1.6803
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 242ms/step - accuracy: 0.4444 - loss: 1.1068
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 172ms/step - accuracy: 0.5833 - loss: 0.9536
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 167ms/step - accuracy: 0.6111 - loss: 0.7712
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 176ms/step - accuracy: 0.8056 - loss: 0.5005
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 170ms/step - accuracy: 0.8889 - loss: 0.4199
Epoch 7/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 169ms/step - accuracy: 0.8889 - loss: 0.3730
Epoch 8/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 172ms/step - accuracy: 0.9444 - loss: 0.2625
Epoch 9/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step - accuracy: 1.0000 - loss: 0.1886
Epoch 10/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 163ms/step - accuracy: 1.0000 - loss: 0.1307
Epoch 11/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 202ms/step - accuracy: 1.0000 - loss: 0.1169
Epoch 12/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 273ms/step - accuracy: 1.0000 - lo

In [ ]:
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

modelo = tf.keras.Sequential([
    data_aug,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

historia = modelo.fit(ds_train, epochs=25)

print('--- EXAMEN FINAL (validación) ---')
resultado = modelo.evaluate(ds_val)
print('Accuracy real:', round(resultado[1]*100, 1), '%')

Epoch 1/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 217ms/step - accuracy: 0.3611 - loss: 1.4253
Epoch 2/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 210ms/step - accuracy: 0.3056 - loss: 1.5475
Epoch 3/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 208ms/step - accuracy: 0.4722 - loss: 1.1955
Epoch 4/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 375ms/step - accuracy: 0.5000 - loss: 1.0024
Epoch 5/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 356ms/step - accuracy: 0.6389 - loss: 0.7113
Epoch 6/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 199ms/step - accuracy: 0.6667 - loss: 0.8643
Epoch 7/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 198ms/step - accuracy: 0.6111 - loss: 0.7868
Epoch 8/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 199ms/step - accuracy: 0.7222 - loss: 0.5085
Epoch 9/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 204ms/step - accuracy: 0.8611 - loss: 0.5320
Epoch 10/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 209ms/step - accuracy: 0.8333 - loss: 0.4578
Epoch 11/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 217ms/step - accuracy: 0.8611 - loss: 0.3335
Epoch 12/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 209ms/step - accuracy: 0.9167 - lo